Multiclass classification with neural networks is about predicting which one of several possible classes an input belongs to (for example, one of 3 flower types), instead of just “yes/no” as in binary classification.



#### Binary vs Multiclass in Neural Networks

- Binary classification: Output is a single number between 0 and 1 (often using a sigmoid activation), interpreted as the probability of the “positive” class.  
- Multiclass classification: You need the model to choose one of many classes and ideally give a probability for each class.

Earlier methods (like “one-vs-rest” or “one-vs-one”) build multiple binary models and combine them. Neural networks *can* use those, but it means training many separate networks, which is expensive.

A more common approach with neural networks is to use one single model that outputs one score per class and then converts those scores into probabilities using softmax.



#### Softmax Output Layer

To adapt a neural network for multiclass classification, you change the output layer:

- Instead of 1 output node (binary), you use one output node per class.  
- This final layer, together with the softmax function, is called a softmax layer.

Process:

1. The network computes raw scores for each class:  
   $$
   x_1, x_2, ..., x_m
   $$  
   where $m$ is the number of classes. These can be any real numbers (positive or negative).

2. The softmax function converts these raw scores into probabilities $y_1, ..., y_m$ such that:  
   - Each $y_i$ is between 0 and 1.  
   - All $y_i$ add up to 1.  
   - The relative order is preserved: if $x_i < x_j$, then $y_i < y_j$.

So softmax is like a multiclass version of sigmoid: it turns arbitrary scores into a probability distribution over classes.

For prediction, you could technically just take the class with the largest raw score $x_i$. Softmax doesn’t change which class is largest; it mainly matters for training and probability interpretation.



#### One-Hot Encoding for Labels

To train the network, you need to provide the true class labels in a form that matches the softmax outputs.

This is done with one-hot encoding:

- If there are $m$ classes, each label is represented as a vector of length $m$.  
- All entries are 0 except for a single 1 at the position of the true class.

For example (conceptually, not tied to the dataset):
- Class A → $[1, 0, 0]$  
- Class B → $[0, 1, 0]$  
- Class C → $[0, 0, 1]$

This encoding expresses that we are 100% certain about the correct class (1 for the true one, 0 for the others).

Softmax outputs a vector of values between 0 and 1 that also sum to 1, so it matches perfectly with these one-hot targets:  
- Each component can be seen as the model’s “certainty” for that class.  
- The training objective tries to make the predicted probability vector close to the one-hot vector.

If you didn’t use softmax and worked directly with raw scores, you would need label values that could range from negative to positive infinity, and you couldn’t clearly express “full certainty” in the same clean probability way.



#### Probabilistic Interpretation and Caution

Because of softmax:

- The output vector from the network can be interpreted as probabilities: e.g., 0.8 for class 1, 0.05 for class 2, 0.15 for class 3.  
- This is convenient for understanding how confident the model is in each possible class.

However:

- These probabilities reflect the model’s internal belief, not necessarily the true real-world probabilities.  
- Neural networks can get stuck in local minima or be miscalibrated, so the numeric values (like “80%”) should not be taken as perfect probabilities, just as indicative confidence.



#### Using Softmax and One-Hot in Practice (High Level)

In practical code with a deep learning library:

1. Prepare labels using one-hot encoding.  
2. Build the network with:
   - Hidden layers (e.g., Dense with ReLU).  
   - Final layer: Dense with number of units equal to the number of classes, activation = `"softmax"`.

3. Compile the model with:
   - A suitable optimizer (variant of stochastic gradient descent).  
   - A loss function that matches softmax + one-hot (typically categorical cross-entropy).  
   - Metrics like accuracy.

4. Train the model on data and one-hot labels for multiple epochs.

During training, softmax + one-hot encoding + cross-entropy loss work together to adjust the network’s weights so that the predicted probability distribution over classes matches the one-hot target as closely as possible.



#### Core Concept Summary

- Multiclass classification with neural networks usually uses one model with a softmax output layer instead of multiple binary models.  
- Softmax converts raw scores for each class into a probability distribution.  
- One-hot encoding represents the true class as a probability vector (1 for the true class, 0 for others).  
- This combination allows the network to:
  - Be trained efficiently with gradient-based methods.  
  - Output interpretable per-class probabilities for multiclass problems.

Different ways to encode class labels for Keras, and how to look at loss instead of accuracy during training.



### 1. Label Encodings and Matching Loss Functions

When you train a neural network for multiclass classification, you must choose how to represent the class labels and which loss function goes with that representation.

#### One‑hot encoding + `categorical_crossentropy`

- Each class is represented as a vector with length equal to the number of classes.  
- All entries are 0 except a single 1 at the position of the true class.  
- In Keras, when your labels are one‑hot encoded, you use the `categorical_crossentropy` loss.

#### Integer encoding + `sparse_categorical_crossentropy`

- Each class is represented by a single integer, like:
  - 0, 1, 2 for three classes.  
- No vectors; just one number per sample.
- In Keras, when your labels are integers, you use `sparse_categorical_crossentropy`.

Key concept:  
Both encodings represent the *same information*, and both loss functions behave equivalently in terms of model performance.  
You just must match them correctly:

- One‑hot labels → `categorical_crossentropy`  
- Integer labels → `sparse_categorical_crossentropy`

If you mix them (e.g., integer labels with `categorical_crossentropy`), Keras will either error or behave incorrectly.



### 2. Plotting Loss vs. Accuracy

When training a neural network, Keras can track different metrics over epochs (training passes), such as:

- Accuracy: How often predictions match the true labels.  
- Loss: How far off the predictions are according to the chosen loss function (e.g., cross‑entropy).

#### Accuracy curves

- Typically increase as training epochs go on.  
- Show how well the model is classifying data in a simple, intuitive way.

#### Loss curves

- Typically decrease as training epochs go on.  
- Give a more detailed view of how the model is improving.
- Even when accuracy looks mostly flat, loss can still be slowly decreasing, indicating finer improvements.

In Keras, if you log training history, you can plot accuracy by using the key `"accuracy"`, or loss by using the key `"loss"`.  
Swapping `"accuracy"` for `"loss"` lets you see how the optimization process is progressing, not just the classification success.



### 3. Overfitting Caveat

In the scenario described:

- The model is trained and evaluated on the same dataset.  
- No separate test or validation set is used.

This means:

- You cannot see overfitting in these plots, because the model is always evaluated on data it has seen.  
- Accuracy and loss may look great, but you don’t know if the model will generalize well to new, unseen data.

In practice, you should usually track both training and validation loss/accuracy to detect overfitting.



#### Core Takeaways

- There are two common label encodings for multiclass in Keras:
  - One‑hot → `categorical_crossentropy`  
  - Integer → `sparse_categorical_crossentropy`  
- They are conceptually equivalent; you just choose the right pair.  
- You can plot accuracy to see classification quality and loss to see optimization progress.  
- Without a separate test/validation set, you cannot detect overfitting from these plots alone.